In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np, pandas as pd, matplotlib, glob
import matplotlib.pyplot as plt
from pathlib import Path

SAVE_DIR = Path('/content/drive/MyDrive/diss-datasets')
SIGLIP = SAVE_DIR / 'siglip2_kaggle'
FIGDIR = Path('/content/drive/MyDrive/diss-figures')
FIGDIR.mkdir(exist_ok=True)

matplotlib.rcParams.update({
    'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'figure.dpi': 120, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'axes.spines.top': False, 'axes.spines.right': False,
})

FS = 256
data = np.load(SAVE_DIR / 'CHBMIT_crosspatient_v2.npy', allow_pickle=True).item()
tr, te = data['train'], data['test']
Xtr, ytr = tr['X'], tr['y'].astype(int)
Xte, yte = te['X'], te['y'].astype(int)
file_te, t0_te = np.asarray(te['file']), te['t0'].astype(int)

if 'patient' in te:
    pat_te = np.asarray(te['patient'])
    src = 'v2 file'
else:
    d = np.load(SIGLIP / 'SigLIP2_CHBMIT_test_scores.npz', allow_pickle=True)
    pat_te = np.asarray(d['patient'])
    src = 'siglip2 score file'

print('test dict keys:', list(te.keys()))
print('patient from:', src)
print('train X', Xtr.shape, '| test X', Xte.shape)
print('train pos/neg', int((ytr==1).sum()), int((ytr==0).sum()))
print('test  pos/neg', int((yte==1).sum()), int((yte==0).sum()))
print('patients', np.unique(pat_te))
print('window', Xte.shape[2]/FS, 's | channels', Xte.shape[1])
print()
print('npy files in diss-datasets:')
for p in sorted(SAVE_DIR.glob('*.npy')):
    print('  ', p.name, f'{p.stat().st_size/1e6:.1f} MB')

In [ ]:
i_sz = int(np.where(yte == 1)[0][0])
i_bg = int(np.where(yte == 0)[0][0])

def plot_window(ax, X, title):
    C, T = X.shape
    t = np.arange(T) / FS
    off = np.percentile(np.abs(X), 99) * 2.2
    for c in range(C):
        ax.plot(t, X[c] + (C - 1 - c) * off, lw=0.4, color='k')
    ax.set_yticks([(C - 1 - c) * off for c in range(0, C, 3)])
    ax.set_yticklabels([f'ch{c+1}' for c in range(0, C, 3)])
    ax.set_xlabel('time (s)')
    ax.set_title(title)
    ax.set_xlim(0, t[-1])

fig, axes = plt.subplots(1, 2, figsize=(9, 5), sharey=True)
plot_window(axes[0], Xte[i_bg], 'Interictal')
plot_window(axes[1], Xte[i_sz], 'Ictal (seizure)')
plt.tight_layout()
plt.savefig(FIGDIR / 'fig3_1_example_windows.pdf')
plt.show()

In [ ]:
def mean_corr(X):
    acc = np.zeros((X.shape[1], X.shape[1]))
    for i in range(X.shape[0]):
        acc += np.nan_to_num(np.corrcoef(X[i]))
    return acc / X.shape[0]

C_sz, C_bg = mean_corr(Xtr[ytr == 1]), mean_corr(Xtr[ytr == 0])
D = C_sz - C_bg

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, M, ttl, cm, vl in [
    (axes[0], C_bg, 'Interictal', 'viridis', (-1, 1)),
    (axes[1], C_sz, 'Ictal', 'viridis', (-1, 1)),
    (axes[2], D, 'Difference (ictal $-$ interictal)', 'RdBu_r',
     (-np.abs(D).max(), np.abs(D).max()))]:
    im = ax.imshow(M, cmap=cm, vmin=vl[0], vmax=vl[1])
    ax.set_title(ttl); ax.set_xlabel('channel')
    if ax is axes[0]: ax.set_ylabel('channel')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(FIGDIR / 'fig3_2_correlation_structure.pdf')
plt.show()

iu = np.triu_indices(Xtr.shape[1], 1)
print('mean off-diagonal, interictal:', round(float(C_bg[iu].mean()), 4))
print('mean off-diagonal, ictal:     ', round(float(C_sz[iu].mean()), 4))
print('largest pairwise shift:       ', round(float(np.abs(D[iu]).max()), 4))

In [ ]:
MERGE = 30

def build_events(y, t0, file):
    event_id = np.full(len(y), -1, dtype=int); nxt = 0
    for f in np.unique(file):
        idx = np.where((file == f) & (y == 1))[0]
        if len(idx) == 0: continue
        order = idx[np.argsort(t0[idx])]
        starts = t0[order]
        loc = np.zeros(len(order), dtype=int)
        for i in range(1, len(order)):
            loc[i] = loc[i-1] + (1 if (starts[i] - starts[i-1]) > MERGE else 0)
        for l, gi in zip(loc, order): event_id[gi] = nxt + l
        nxt += int(loc.max()) + 1
    return event_id, nxt

event_id, n_events = build_events(yte, t0_te, file_te)
ev_pat = np.array([pat_te[event_id == e][0] for e in range(n_events)])
print('events:', n_events)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={'width_ratios': [2.2, 1]})
pats = list(np.unique(pat_te))
ax = axes[0]
for r, p in enumerate(pats):
    m = pat_te == p
    tt, yy = t0_te[m] / 3600, yte[m]
    ax.scatter(tt[yy == 0], np.full((yy == 0).sum(), r), s=3, c='lightgrey', marker='|')
    ax.scatter(tt[yy == 1], np.full((yy == 1).sum(), r), s=18, c='crimson', marker='|')
ax.set_yticks(range(len(pats))); ax.set_yticklabels(pats)
ax.set_xlabel('recording time (h)')
ax.set_title('Test windows over time (red = ictal)')

ax = axes[1]
counts = [int((ev_pat == p).sum()) for p in pats]
ax.barh(range(len(pats)), counts, color='crimson', alpha=0.75)
ax.set_yticks(range(len(pats))); ax.set_yticklabels(pats)
ax.set_xlabel('seizure events'); ax.set_title(f'Events per patient (n = {n_events})')
for i, c in enumerate(counts): ax.text(c + 0.2, i, str(c), va='center', fontsize=8)
plt.tight_layout()
plt.savefig(FIGDIR / 'fig3_3_event_structure.pdf')
plt.show()

In [ ]:
rng = np.random.default_rng(0)
W = Xte[i_sz]; C, T = W.shape

W_perm = W[rng.permutation(C)]
W_desync = np.empty_like(W)
for c, s in enumerate(rng.integers(-T // 4, T // 4, size=C)):
    W_desync[c] = np.roll(W[c], s)
W_pool = np.repeat(W.mean(axis=0, keepdims=True), C, axis=0)

fig, axes = plt.subplots(1, 4, figsize=(13, 4.6), sharey=True)
for ax, M, ttl in [
    (axes[0], W, 'Intact'),
    (axes[1], W_perm, 'Permuted\n(channel identity destroyed)'),
    (axes[2], W_desync, 'Desynchronised\n(cross-channel timing destroyed)'),
    (axes[3], W_pool, 'Pooled\n(all cross-channel structure destroyed)')]:
    plot_window(ax, M, ttl)
plt.tight_layout()
plt.savefig(FIGDIR / 'fig4_1_ablations.pdf')
plt.show()

In [ ]:
def to_image(sig, n_seg=16):
    L = (len(sig) // n_seg) * n_seg
    img = sig[:L].reshape(n_seg, -1)
    return (img - img.min()) / (img.max() - img.min() + 1e-9)

fig = plt.figure(figsize=(12, 4.4))
gs = fig.add_gridspec(3, 12, width_ratios=[1]*6 + [0.25] + [1]*5)
for k in range(6):
    ax = fig.add_subplot(gs[k // 3, k % 3])
    ax.imshow(to_image(W[k]), cmap='gray', aspect='auto')
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f'ch{k+1}', fontsize=7)
ax = fig.add_subplot(gs[2, 0:3])
ax.text(0.5, 0.5, f'... {C} separate images,\nembeddings concatenated',
        ha='center', va='center', fontsize=8); ax.axis('off')
ax = fig.add_subplot(gs[:, 7:])
ax.imshow(np.vstack([to_image(W[c], n_seg=4) for c in range(C)]), cmap='gray', aspect='auto')
ax.set_xticks([]); ax.set_yticks([])
ax.set_title('Stacked: all channels, one image', fontsize=9)
fig.text(0.16, 0.98, 'Per-channel: one image per channel', ha='center', fontsize=9)
plt.savefig(FIGDIR / 'fig4_2_imaging_strategies.pdf')
plt.show()

In [ ]:
f0 = file_te[np.where(yte == 1)[0][0]]
m = file_te == f0
tt, yy, ee = t0_te[m], yte[m], event_id[m]
o = np.argsort(tt); tt, yy, ee = tt[o], yy[o], ee[o]

fig, ax = plt.subplots(figsize=(11, 2.6))
for t, lab in zip(tt, yy):
    ax.add_patch(plt.Rectangle((t, 0.2), 9, 0.6,
                 facecolor='crimson' if lab else 'lightgrey', edgecolor='white', lw=0.5))
for e in np.unique(ee[ee >= 0]):
    s = tt[ee == e]
    ax.annotate('', xy=(s.min(), 1.05), xytext=(s.max() + 10, 1.05),
                arrowprops=dict(arrowstyle='<->', color='navy', lw=1.2))
    ax.text((s.min() + s.max()) / 2, 1.15, f'event {e}', ha='center', fontsize=8, color='navy')
ax.set_xlim(tt.min() - 30, tt.max() + 60); ax.set_ylim(0, 1.45)
ax.set_yticks([]); ax.set_xlabel('time within recording (s)')
ax.set_title(f'Window labels and event grouping, {f0} (merge gap $\\leq$ {MERGE} s)')
plt.tight_layout()
plt.savefig(FIGDIR / 'fig4_3_event_scoring.pdf')
plt.show()

In [ ]:
rows = [{'Dataset': 'CHB-MIT', 'Domain': 'Clinical EEG', 'Channels': Xte.shape[1],
         'Length': Xte.shape[2], 'Classes': 2, 'Train': len(ytr), 'Test': len(yte),
         'Split': 'cross-patient'}]

for name in ['CMJ', 'MP8', 'MP50', 'SYNTH', 'Rowing']:
    d = np.load(SAVE_DIR / f'{name}.npy', allow_pickle=True).item()
    a, b = d['train'], d['test']
    rows.append({'Dataset': name,
                 'Domain': 'Synthetic' if name == 'SYNTH' else 'Motion capture',
                 'Channels': a['X'].shape[1], 'Length': a['X'].shape[2],
                 'Classes': len(np.unique(a['y'])), 'Train': len(a['y']),
                 'Test': len(b['y']), 'Split': 'predefined'})

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print()
print(df.to_latex(index=False, escape=False,
                  caption='Datasets used in this thesis.', label='tab:datasets'))

In [ ]:
for n in ['CMJ', 'MP8', 'MP50', 'SYNTH', 'Rowing']:
    a = np.load(SAVE_DIR / f'{n}_text_embeddings_train.npy')
    b = np.load(SAVE_DIR / f'{n}_text_embeddings_test.npy')
    print(f'{n:7s} train {str(a.shape):16s} test {str(b.shape):16s} '
          f'nan {int(np.isnan(a).sum())} const {bool(np.allclose(a.std(0), 0))}')

x8 = np.load(SAVE_DIR / 'MP8_text_embeddings_test.npy')
x50 = np.load(SAVE_DIR / 'MP50_text_embeddings_test.npy')
print()
print('MP8 and MP50 text embeddings identical:',
      x8.shape == x50.shape and np.array_equal(x8, x50))

print()
for tag in ['', 'permuted_', 'desync_', 'chanpool_']:
    p = SAVE_DIR / f'Chronos2_CHBMIT_v2_{tag}test_emb.npy'
    print(f"{tag or 'intact':10s}", np.load(p, mmap_mode='r').shape)

In [ ]:
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, f1_score

x8 = np.load(SAVE_DIR / 'MP8_text_embeddings_test.npy')
x50 = np.load(SAVE_DIR / 'MP50_text_embeddings_test.npy')
print('MP8 vs MP50 mean abs diff:', float(np.abs(x8 - x50).mean()))
print('rows identical:', int((np.abs(x8 - x50).sum(1) == 0).sum()), 'of', len(x8))
print()

out = []
for n in ['CMJ', 'MP8', 'MP50', 'SYNTH', 'Rowing']:
    d = np.load(SAVE_DIR / f'{n}.npy', allow_pickle=True).item()
    ya, yb = d['train']['y'], d['test']['y']
    Za = np.load(SAVE_DIR / f'{n}_text_embeddings_train.npy')
    Zb = np.load(SAVE_DIR / f'{n}_text_embeddings_test.npy')
    if len(ya) != len(Za) or len(yb) != len(Zb):
        print(f'{n}: length mismatch, labels {len(ya)}/{len(yb)} '
              f'embeddings {len(Za)}/{len(Zb)}')
        continue
    p = RidgeClassifier().fit(Za, ya).predict(Zb)
    out.append({'dataset': n, 'channels': d['train']['X'].shape[1],
                'acc': round(accuracy_score(yb, p), 4),
                'macro_f1': round(f1_score(yb, p, average='macro'), 4)})

print(pd.DataFrame(out).to_string(index=False))

In [ ]:
from sklearn.linear_model import RidgeClassifierCV
from sklearn.preprocessing import StandardScaler

out = []
for n in ['CMJ', 'MP8', 'MP50', 'SYNTH', 'Rowing']:
    d = np.load(SAVE_DIR / f'{n}.npy', allow_pickle=True).item()
    ya, yb = d['train']['y'], d['test']['y']
    Za = np.load(SAVE_DIR / f'{n}_text_embeddings_train.npy')
    Zb = np.load(SAVE_DIR / f'{n}_text_embeddings_test.npy')
    sc = StandardScaler().fit(Za)
    clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 13)).fit(sc.transform(Za), ya)
    p = clf.predict(sc.transform(Zb))
    vals, cnt = np.unique(yb, return_counts=True)
    out.append({'dataset': n, 'channels': d['train']['X'].shape[1],
                'classes': len(vals), 'majority': round(cnt.max()/len(yb), 4),
                'acc': round(accuracy_score(yb, p), 4),
                'macro_f1': round(f1_score(yb, p, average='macro'), 4),
                'alpha': float(clf.alpha_)})

print(pd.DataFrame(out).to_string(index=False))

In [ ]:
!pip install -q pymupdf

import fitz, shutil
from datetime import datetime

for pdf in sorted(FIGDIR.glob('*.pdf')):
    doc = fitz.open(pdf)
    doc[0].get_pixmap(dpi=150).save(FIGDIR / f'{pdf.stem}.png')
    doc.close()
    print('png:', pdf.stem)

RESULTS = '/content/drive/MyDrive/results.xlsx'
try:
    df_abba = pd.DataFrame(out)
    shutil.copy(RESULTS, RESULTS.replace('.xlsx', f'_backup_{datetime.now():%Y%m%d_%H%M}.xlsx'))
    with pd.ExcelWriter(RESULTS, engine='openpyxl', mode='a', if_sheet_exists='replace') as w:
        df_abba.to_excel(w, sheet_name='LLM-ABBA', index=False)
    print('\nsheets:', pd.ExcelFile(RESULTS).sheet_names)
except NameError:
    print('\nskipped xlsx: run the regularised LLM-ABBA cell first')

!cd /content/drive/MyDrive && rm -f diss-figures.zip && zip -qr diss-figures.zip diss-figures -x "*.pdf"
print('\nzip ready at MyDrive/diss-figures.zip')